# Adaptive Residual Steering (ARS) — Kaggle Execution Runner

This notebook provides a lightweight, reproducible Kaggle entrypoint for **Adaptive Residual Steering (ARS)** on frozen language models.

### Pipeline Overview
1. Clone or sync repository code
2. Install minimal dependencies
3. Load configuration and model
4. Execute staged training or evaluation
5. Export checkpoints & benchmark summaries

In [ ]:
# Environment setup & dependency installation
!pip install -q transformers datasets accelerate

In [ ]:
import sys
import os

# Ensure project root is in sys.path
sys.path.append(os.path.abspath("."))

import torch
from src.configs.config import RSCConfig
from src.models.wrapper import Phi2WithRSC
from src.data.dataset import gsm8k_to_cot
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {DEVICE}")

In [ ]:
cfg = RSCConfig()
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
).to(DEVICE)

model = Phi2WithRSC(base_model, cfg, device=DEVICE)
print(f"Wrapped model ready with {len(model.target_layers)} candidate layers: {model.target_layers}")

In [ ]:
# Run Phase 1.5 Router Bootstrap via CLI runner or direct module API
!python scripts/run_experiment.py --phase bootstrap --epochs 3